In [ ]:
from functools import partial
from notebooks._utils import calculate_series_ensemble_accuracy
from notebooks._utils import calculate_parallel_ensemble_accuracy

ds_name = "myriadlama-debug"
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"

def get_layers(model: str):
    if model.startswith("llama3.2_1b"):
        layers = [8, 10, 12, 14, 16]
    elif model.startswith("llama3.2_3b"):
        layers = [16, 19, 22, 25, 28]
    elif model.startswith("llama3.1_8b"):
        layers = [16, 20, 24, 28, 32]
    elif model.startswith("qwen2.5_3b"):
        layers = [20, 24, 28, 32, 36]
    elif model.startswith("qwen2.5_7b"):
        layers = [16, 19, 22, 25, 28]
    elif model.startswith("qwen2.5_14b"):
        layers = [24, 30, 36, 42, 48]
    else:
        raise NotImplementedError(f"Layers not defined for model {model}")
    return layers


In [57]:
num_fewshots = 0

# for model_name in ["llama3.2_1b", "llama3.2_3b", "llama3.1_8b", "qwen2.5_3b", "qwen2.5_7b", ]:
for model_name in ["llama3.2_1b_it", "llama3.2_3b_it", "llama3.1_8b_it", "qwen2.5_3b_it", "qwen2.5_7b_it"]:
# # for model_name in ["llama3.1_8b"]:
# model_name = "llama3.1_8b"
    print(f"\n=================== Model: {model_name} ===================")
    dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."

    report_acc_base_setting = partial(
        calculate_parallel_ensemble_accuracy, 
        dump_file_prefix=dump_file_prefix, repeat_paras=False,
        num_paraphrases=5, num_fewshots=num_fewshots, use_generation=True)

    print("---- Calculating baseline ----")
    df = calculate_series_ensemble_accuracy(
        dump_file_prefix=dump_file_prefix, 
        single_para_qapair=True, explicit_prompts=False, repeat_paras=False, 
        modifyattn=False, modifyrope=False, scale_score=False, 
        num_paraphrases=1, num_fewshots=num_fewshots)


    print("\n---- Logits-based Ensemble (Average) ----")
    report_acc_base_setting(logits_ensemble_method="avg")

    print("\n---- Logits-based Ensemble (Maximum) ----")
    report_acc_base_setting(logits_ensemble_method="max")

    single_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="layer_output_avg", multilayer=False)
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="layer_output_avg", multilayer=True)

    single_ffnavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_avg", multilayer=False)
    multip_ffnavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_avg", multilayer=True)

    single_ffnmax_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_max", multilayer=False)
    multip_ffnmax_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_max", multilayer=True)

    # for alpha in [0.25, 0.5, 0.75, 1.0]:
    for alpha in [1.0]:
        for token_mode in ["last"]:
            for layer in get_layers(model_name):
                # print(f"\n---- Single Layer Avg Ensemble (token_mode={token_mode}, layer={layer}) ----")
                # single_layavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)
                print(f"\n---- Multi Layer Avg Ensemble (token_mode={token_mode}) Layer-{layer} Alpha-{alpha} ----")
                multip_layavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)
                
                print(f"\n---- Single FFN Avg Ensemble (token_mode={token_mode}, layer={layer}) ----")
                multip_ffnavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)



=================== Model: llama3.2_1b_it ===================
---- Calculating baseline ----
File ./singleparaqapair.0fshots.5samples.1paras.feather does not exist!

---- Logits-based Ensemble (Average) ----
Acc: 0.2180 ==> 🏷️ 5paras 0shots  None layerNone  alpha1.0 token-all

---- Logits-based Ensemble (Maximum) ----
Acc: 0.2940 ==> 🏷️ 5paras 0shots  None layerNone  alpha1.0 token-all

---- Multi Layer Avg Ensemble (token_mode=last) Layer-8 Alpha-1.0 ----
Acc: 0.2900 ==> 🏷️ 5paras 0shots  layer_output_avg layer8 Multilayer alpha1.0 token-last

---- Single FFN Avg Ensemble (token_mode=last, layer=8) ----
Acc: 0.2650 ==> 🏷️ 5paras 0shots  ffn_activation_avg layer8 Multilayer alpha1.0 token-last

---- Multi Layer Avg Ensemble (token_mode=last) Layer-10 Alpha-1.0 ----
Acc: 0.2830 ==> 🏷️ 5paras 0shots  layer_output_avg layer10 Multilayer alpha1.0 token-last

---- Single FFN Avg Ensemble (token_mode=last, layer=10) ----
Acc: 0.2670 ==> 🏷️ 5paras 0shots  ffn_activation_avg layer10 Multilaye

In [55]:
# for alpha in [0.25, 0.5, 0.75, 1.0]:
for alpha in [1.0]:
    for token_mode in ["last"]:
        for layer in get_layers(model_name):
            # print(f"\n---- Single Layer Avg Ensemble (token_mode={token_mode}, layer={layer}) ----")
            # single_layavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)
            print(f"\n---- Multi Layer Avg Ensemble (token_mode={token_mode}) Layer-{layer} Alpha-{alpha} ----")
            multip_layavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)
            
            print(f"\n---- Single FFN Avg Ensemble (token_mode={token_mode}, layer={layer}) ----")
            multip_ffnavg_report(token_mode=token_mode, ensemble_alpha=alpha, ensemble_layer=layer)


---- Multi Layer Avg Ensemble (token_mode=last) Layer-16 Alpha-1.0 ----
Acc: 0.5140 ==> 🏷️ 5paras 5shots  layer_output_avg layer16 Multilayer alpha1.0 token-last

---- Single FFN Avg Ensemble (token_mode=last, layer=16) ----
Acc: 0.5210 ==> 🏷️ 5paras 5shots  ffn_activation_avg layer16 Multilayer alpha1.0 token-last

---- Multi Layer Avg Ensemble (token_mode=last) Layer-19 Alpha-1.0 ----
Acc: 0.5160 ==> 🏷️ 5paras 5shots  layer_output_avg layer19 Multilayer alpha1.0 token-last

---- Single FFN Avg Ensemble (token_mode=last, layer=19) ----
Acc: 0.5200 ==> 🏷️ 5paras 5shots  ffn_activation_avg layer19 Multilayer alpha1.0 token-last

---- Multi Layer Avg Ensemble (token_mode=last) Layer-22 Alpha-1.0 ----
Acc: 0.5160 ==> 🏷️ 5paras 5shots  layer_output_avg layer22 Multilayer alpha1.0 token-last

---- Single FFN Avg Ensemble (token_mode=last, layer=22) ----
Acc: 0.5240 ==> 🏷️ 5paras 5shots  ffn_activation_avg layer22 Multilayer alpha1.0 token-last

---- Multi Layer Avg Ensemble (token_mode=las